# Job Postings: Sample, Methods and Limitations

The reference document behind [What Firms Ask For](./_job_postings_findings_ipynb.ipynb).
Everything the findings page asserts is derived here, and everything it defers
is spelled out here: the sample frame, what each way of counting over- and
under-states, the rules that decide what counts as a mention, and the claims
this data cannot support.

Both notebooks load the same committed tables through `jobpostings.report` and
neither recomputes anything, so the two cannot drift apart on a number.

Nothing here touches the network. Collection is a separate manual step
(`src/collect_parallel.py`); this notebook consumes what that step committed,
which is what makes the figures reproducible from a clone.

In [1]:
from jobpostings import report

d = report.load()
report.headline(d)

collection runs: 2026-09-25, 2026-09-27
11,346 postings from 64 firms
9,237 technical postings
1,600 mention a pipeline, ETL/ELT or orchestration


## What the sample is, and what it is not

The sample frame is `data_manual/manifest.yml`: firms whose careers systems
expose a machine-readable endpoint we have found and validated. That is a real
selection rule with real consequences.

- It is **not a random sample** of quantitative finance. It over-represents
  firms that publish openly and under-represents firms that recruit through
  networks and headhunters. Coverage is systematically worse for banks, which
  run fragmented legacy applicant-tracking systems.
- Postings measure **hiring intent**, not deployed infrastructure. A firm can
  run Airflow for a decade and never name it; a firm can name a tool it is only
  beginning to adopt.
- Firms differ in how specific they are. Some publish their whole stack; others
  name almost nothing. **A zero is ambiguous** between "does not use it" and
  "does not say".

Those are not boilerplate. They are the difference between a defensible reading
and an overreach, and the frame below is printed in full so they can be checked.

## The sample frame

One row per firm. Firms with zero postings are kept rather than dropped: they
are either targets whose route is unresolved, firms whose own system was failing
when we last collected, or firms with no machine-readable board at all. Keeping
them visible preserves the denominator of *attempted* coverage.

In [2]:
frame = d["frame"]
collected = frame[frame["n_postings"] > 0]
pending = frame[frame["n_postings"] == 0]

view = collected[
    ["firm_name", "firm_type", "route", "sampling", "stored_scope",
     "n_postings", "n_technical", "n_pipeline"]
].sort_values("n_postings", ascending=False)
view.columns = [
    "Firm", "Type", "Route", "Sampling", "Stored scope",
    "Postings", "Technical", "Pipeline-mentioning",
]
view.reset_index(drop=True)

,Firm,Type,Route,Sampling,Stored scope,Postings,Technical,Pipeline-mentioning
0,JPMorganChase,bank,oracle_recruiting,census,technical_only,2425,2425,501
1,HSBC,bank,eightfold,census,all_postings,1472,176,38
2,Citi,bank,workday,census,technical_only,1023,1023,178
3,Goldman Sachs,bank,oracle_recruiting,census,technical_only,559,559,63
4,Barclays,bank,workday,census,technical_only,421,421,80
5,Morgan Stanley,bank,workday,census,technical_only,400,400,50
6,Deutsche Bank,bank,workday,census,technical_only,361,361,46
7,Jane Street,prop_trading,greenhouse,census,all_postings,230,150,4
8,Bank of America,bank,workday,census,technical_only,220,220,48
9,Point72 (incl. Cubist Systematic Strategies),multistrat,greenhouse,census,all_postings,220,166,23


`Stored scope` matters when comparing the Postings column across firms. Some
adapters (Workday, Oracle) filter to technical titles before storing; the rest
store the whole board. So `n_postings` is not comparable across routes, while
`n_technical` is. The analysis uses technical denominators throughout.

In [3]:
print(f"{len(collected)} firms collected, {len(pending)} in the frame with no data\n")
for _, r in pending.iterrows():
    print(f"  {r['status']:12} {r['firm_name']}")

64 firms collected, 9 in the frame with no data

  unresolved   Acadian Asset Management
  broken       Balyasny Asset Management
  unresolved   BNP Paribas
  unsamplable  Capula Investment Management
  unsamplable  Hudson Bay Capital
  unresolved   Rokos Capital Management
  unsamplable  Susquehanna International Group
  unresolved   State Street Global Advisors
  unsamplable  UBS


Statuses are deliberately distinguished, because they would otherwise all read
as "no data":

- **unresolved** — not found yet. Says nothing about the firm. Discovery is
  non-deterministic: one firm resolved on one run and came back "no
  machine-readable board" on the next with its endpoint working the whole time.
- **unsamplable** — looked at, no route we can use. Mostly this means *we have
  not written the adapter*, not that the firm cannot be reached; that
  distinction is recorded per firm.
- **broken** — the route works in principle and the firm's own system is
  failing.

Per-firm caveats live in the manifest and travel into the frame, so a count and
its caveat cannot drift apart.

In [4]:
for _, r in frame[frame["limitations"] != ""].iterrows():
    print(f"{r['firm_name']} ({r['status']}, n={r['n_postings']})\n  {r['limitations']}\n")

AQR Capital Management (resolved, n=51)
  Postings name essentially no infrastructure products.

Arrowstreet Capital (resolved, n=19)
  POST {"appliedFacets":{},"limit":20,"offset":0,"searchText":""} returned total=23 and requires offset paging (20 per page); a second site, Campus_Careers, on the same host/tenant holds 4 additional student/intern postings and must be queried separately to get full coverage.

Balyasny Asset Management (broken, n=0)
  Workday tenant returns HTTP 401 "Unable to verify credentials for system account 39$1476" on the CXS API and serves a "Workday is currently unavailable" page for individual roles. Failure is on the firm's side, not a block on us. Retry on each run.

Bank of America (resolved, n=220)
  POST to the CXS endpoint returns total=1950 with 20 postings per page (paginate via offset); this Workday site is experienced/lateral hiring only — campus/university roles live on a separate Avature board (bankcampuscareers.tal.net), and no other ghr site name

## Three denominators

A share is meaningless without its denominator, and the choice moves the answer
by a factor of five.

- **all_technical** — every posting with a technical title. Wide; it includes
  low-latency C++ and trading roles that would never name an orchestrator, so it
  *understates* prevalence among the people who build pipelines.
- **pipeline_mentioning** — technical postings whose text mentions a pipeline,
  ETL/ELT or orchestration. Much closer to the population the course is about,
  but selected on the outcome's own vocabulary, so it *overstates* prevalence
  relative to engineering hiring generally.
- **pipeline_distinct_roles** — the same pool with one row per (firm, title), so
  a role advertised in six offices counts once. The robustness check.

In [5]:
summary = d["summary"]
denoms = (
    summary[summary["firm_type"] == "ALL"][["denominator", "denominator_n", "n_firms_in_pool"]]
    .drop_duplicates()
    .set_index("denominator")
)
denoms

,denominator_n,n_firms_in_pool
denominator,,
all_technical,9237,63
pipeline_distinct_roles,1349,57
pipeline_mentioning,1600,57


## Excluding trading roles: what it costs

The technical-title filter excludes trading roles. That is a choice, so here is
what it discards: postings with a trading title that fail the filter, and how
many tool mentions are inside them.

In [6]:
import re

idx, long = d["idx"], d["long"]
TRADE = re.compile(r"\btrad(?:er|ing)\b|\bmarket mak", re.IGNORECASE)
excluded = idx[(~idx["is_technical"]) & (idx["title"].str.contains(TRADE))]
ids = set(excluded["posting_id"])
inside = long[long["posting_id"].isin(ids)]
print(f"postings with a trading title excluded by the filter: {len(excluded)}")
print(f"  as a share of all {len(idx):,} postings: {len(excluded) / len(idx):.1%}")
print(f"  tool mentions inside them: {len(inside)}")
if len(inside):
    print(inside.groupby("display").size().sort_values(ascending=False).head(8).to_string())

postings with a trading title excluded by the filter: 157
  as a share of all 11,346 postings: 1.4%
  tool mentions inside them: 184
display
Python                                      77
SQL (any dialect)                           34
R                                           16
C++                                         12
Data-quality / validation work (generic)    10
Bloomberg                                    7
MATLAB                                       5
kdb+ / q                                     5


## Four ways of counting, and why the choice matters more than the sample size

Firms differ in board size by three orders of magnitude, from a handful of
postings to several thousand. So "what share of postings name this tool" is
largely a statement about whoever posts the most.

- **n_postings / n_firms** — raw counts, no weighting. Look at these first.
- **% firms naming it** — the extensive margin: the fraction of firms with at
  least one such posting. The most assumption-light. A firm cannot inflate it by
  posting more, by reposting one role across offices, or by repeating a stack
  blurb. It cannot distinguish one mention from many.
- **% postings (pooled)** — total mentions over total postings. Dominated by the
  largest boards.
- **% postings (firm mean)** — equal weight per firm. Immune to board size, but
  a two-posting firm reports 0% or 50% as though either were a measurement.
- **% postings (shrunk)** — the firm mean with each rate pulled toward the
  pooled rate in proportion to how little data the firm has (beta-binomial
  empirical Bayes).

`index/firm_rates.csv` holds the numerator and denominator behind every firm, so
the shrinkage is inspectable rather than a black box.

In [7]:
report.tools_table(d["summary"], layers=report.SCHEDULING_LAYERS)

,Tool,Layer,Postings,Firms,Firms in pool,% firms naming it,% postings (shrunk),% postings (firm mean),% postings (pooled)
0,Apache Airflow,orchestrator,164,33,57,57.9,13.1,13.8,10.2
1,Dagster,orchestrator,23,14,57,24.6,1.8,3.7,1.4
2,Slurm,hpc_scheduler,26,12,57,21.1,2.2,4.1,1.6
3,Argo Workflows / Argo CD,durable_workflow,23,10,57,17.5,1.5,1.6,1.4
4,CA AutoSys,enterprise_scheduler,28,9,57,15.8,1.7,0.9,1.8
5,Prefect,orchestrator,13,9,57,15.8,1.1,3.5,0.8
6,AWS Step Functions,durable_workflow,42,8,57,14.0,2.3,1.3,2.6
7,BMC Control-M,enterprise_scheduler,25,8,57,14.0,1.8,1.6,1.6
8,Azure Data Factory,durable_workflow,10,6,57,10.5,0.6,0.4,0.6
9,Temporal.io,durable_workflow,20,5,57,8.8,1.1,0.4,1.2


### The estimators, demonstrated

One firm with a single posting that names the tool, against one firm with a
thousand postings that never does. Watch what each estimator does as the small
firm gains data.

In [8]:
import pandas as pd

from jobpostings.index import _estimators

rows = []
for size in (1, 10, 100, 1000):
    e = _estimators(pd.Series([size, 0]), pd.Series([size, 1000]))
    rows.append(
        {
            "mentioning firm's postings": size,
            "% firms naming it": round(e["firms_any"] * 100, 1),
            "pooled": round(e["pooled"] * 100, 2),
            "firm mean": round(e["firm_mean"] * 100, 2),
            "shrunk": round(e["shrunk"] * 100, 2),
        }
    )
pd.DataFrame(rows)

,mentioning firm's postings,% firms naming it,pooled,firm mean,shrunk
0,1,50.0,0.10,50.0,2.43
1,10,50.0,0.99,50.0,17.01
2,100,50.0,9.09,50.0,42.51
3,1000,50.0,50.00,50.0,50.00


The firm mean always gives the one-posting firm half the vote. Pooled almost
never does. The shrunk estimate earns its way from one to the other as evidence
accumulates. The extensive margin sits at 50% throughout, because it only asks
*whether* a firm names the tool, never how loudly.

One property worth stating because it is easy to assume otherwise: the shrunk
estimate does **not** have to lie between the firm mean and the pooled rate. It
is a per-firm convex combination that is then averaged, so when firm size
correlates with a firm's own rate it can sit outside that interval. It does so
for a small number of rows in this corpus, by small margins.

### A live test: does the weighting hold when one firm gets huge?

Collection was briefly capped at 250 postings per firm as a storage budget. When
the cap was removed, one firm's contribution grew roughly tenfold and the corpus
grew by half. That is a natural experiment on the estimators, and the results
are recorded here because they are the justification for the weighting scheme:

| estimator | mean absolute move | max | rows moving >1pp |
|---|---|---|---|
| % postings (firm mean) | 0.078pp | 0.474pp | 0 |
| % postings (shrunk) | 0.410pp | 3.067pp | 9 |
| % postings (pooled) | 0.881pp | 7.319pp | 15 |
| % firms naming it | 1.025pp | 5.263pp | 32 |

Equal weight per firm was essentially immune: not one row moved by a percentage
point. Pooled moved as a volume-weighted statistic must, shifting toward the new
firm's profile — Java +7.3pp, CI/CD +6.9, Kubernetes +5.1, Terraform +4.8 — none
of which reflects a change in the market.

The extensive margin moved most, which refutes a claim made earlier in this
work. It is immune to *volume inflation of a mention already counted*, but not
to *discovering a mention that truncation had hidden*. 32 tools gained firms and
none lost any: a strictly one-directional correction, so the cap had been
biasing the firm count downward. Capping is now opt-in and unused.

## Who names no orchestrator at all

Not random, and not a gap in the data. Latency-first proprietary shops describe
building their own scheduling layer instead of adopting a product.

In [9]:
ORCH = ["airflow", "prefect", "dagster", "luigi", "control_m", "autosys"]
per_firm = (
    long[long["tool"].isin(ORCH)]
    .pivot_table(index="firm", columns="tool", values="posting_id", aggfunc="nunique")
    .reindex(columns=ORCH)
    .fillna(0)
    .astype(int)
)
tech = idx[idx["denom_all_technical"]].groupby("firm").size().rename("technical")
names = frame.set_index("firm")[["firm_name", "firm_type"]]
table = names.join(tech, how="right").join(per_firm).fillna(0)
for c in table.columns:
    if c not in ("firm_name", "firm_type"):
        table[c] = table[c].astype(int)
none_named = table[table[ORCH].sum(axis=1) == 0]
substantial = none_named[none_named["technical"] >= 20].sort_values(
    "technical", ascending=False
)
print("Firms with 20+ technical postings naming no orchestration product:\n")
for _, r in substantial.iterrows():
    print(f"  {r['firm_name']:30} {r['technical']:5d} technical  ({r['firm_type']})")
print(f"\n{int(substantial['technical'].sum()):,} technical postings at those firms.")

Firms with 20+ technical postings naming no orchestration product:

  Jane Street                      150 technical  (prop_trading)
  Citadel                          108 technical  (multistrat)
  Jump Trading                      94 technical  (prop_trading)
  PIMCO                             84 technical  (asset_manager)
  Hudson River Trading              62 technical  (prop_trading)
  Two Sigma                         43 technical  (systematic)
  The D. E. Shaw Group              42 technical  (systematic)
  AQR Capital Management            36 technical  (systematic)
  Akuna Capital                     33 technical  (market_maker)
  Old Mission Capital               29 technical  (market_maker)

681 technical postings at those firms.


## Every other layer

The full ranking for each layer, so the syllabus can be argued from the data
rather than from the orchestration slice alone.

In [10]:
LAYERS = [
    "transform", "datastore", "streaming", "storage_format", "container",
    "ci", "data_quality", "mlops", "ingestion", "vendor_data", "language",
]
for layer in LAYERS:
    t = report.tools_table(d["summary"], layers=[layer])
    if t.empty:
        continue
    print(f"\n=== {layer} ===")
    print(
        t[["Tool", "Postings", "Firms", "% firms naming it", "% postings (shrunk)"]]
        .to_string(index=False)
    )


=== transform ===
        Tool  Postings  Firms  % firms naming it  % postings (shrunk)
Apache Spark       297     28               49.1                 14.6
      pandas        98     26               45.6                  7.2
       NumPy        69     20               35.1                  4.8
         dbt        50     14               24.6                  3.3
         Ray        31     11               19.3                  2.6
        Dask        21     11               19.3                  1.6
      Polars        20     11               19.3                  1.6
      DuckDB         5      3                5.3                  0.4

=== datastore ===
             Tool  Postings  Firms  % firms naming it  % postings (shrunk)
SQL (any dialect)       599     46               80.7                 37.5
       PostgreSQL       162     32               56.1                 12.5
        Snowflake       195     25               43.9                 11.7
          MongoDB       104     

## How a mention is decided

Every feature comes from a declarative rule in `src/tool_rules.yml`. The rules
carry context tests because a naive keyword search is wrong in ways that change
conclusions, and they are unit-tested against the exact strings that fooled
earlier versions.

The clearest case: "airflow" is also a facilities term. At two firms in this
corpus the only matches were data-centre cooling roles, and counting those would
have put a firm that never names an orchestrator at the top of the Airflow list.

In [11]:
rules = (d["root"] / "src" / "tool_rules.yml").read_text()
start, end = rules.index("  airflow:"), rules.index("  dagster:")
print(rules[start:end].rstrip())

  airflow:
    display: Apache Airflow
    layer: orchestrator
    patterns: ['\bairflow\b']
    exclude_if_context:
      - 'cooling'
      - '\brack(-|\s)?level\b'
      - '\bracks?\b'
      - 'CRAH|CRAC|HVAC'
      - 'airflow management'
      - 'liquid cooling'
      - 'containment'
      - 'data cent(er|re)'
      - '\bCDUs?\b|\bRDHx\b'
    note: >-
      "Airflow" is also a facilities term. Jane Street's and HRT's only matches
      were data-centre cooling roles ("redundancy topologies, containment,
      airflow management", "rack-level cooling and airflow"). Excluding those is
      the difference between Jane Street appearing to be an Airflow shop and the
      truth, which is that it never names an orchestrator.


Other traps the rules guard against, each found by getting it wrong first:
`scalable` matching Scala, `array` matching Ray, "temporal data" matching
Temporal.io, a stack trace matching FINRA TRACE, Monte Carlo simulation matching
a data-observability vendor, and Yarn the JavaScript package manager matching
Hadoop YARN.

## Worked evidence

Counts are only as good as the sentences behind them. These are pulled
programmatically from the corpus, so they cannot drift from the numbers above.

In [12]:
import json


def quotes(tool: str, limit: int = 6, window: int = 150) -> None:
    """Print real sentences behind a tool's count, at most one per firm."""
    hits = set(long[long["tool"] == tool]["posting_id"])
    meta = idx.set_index("posting_id")
    seen, shown = set(), 0
    for path in sorted((d["data_dir"] / "postings").glob("*.jsonl")):
        for line in path.read_text(encoding="utf-8").splitlines():
            if shown >= limit:
                return
            rec = json.loads(line)
            if rec["posting_id"] not in hits or rec["firm"] in seen:
                continue
            m = re.search(rf"\b{tool}\b", rec["text"], re.IGNORECASE)
            if not m:
                continue
            seen.add(rec["firm"])
            lo, hi = max(0, m.start() - window), min(len(rec["text"]), m.end() + window)
            firm = meta.loc[rec["posting_id"], "firm_name"]
            print(f"[{firm} — {rec['title']}]\n  ...{rec['text'][lo:hi]}...\n")
            shown += 1


quotes("airflow", limit=6)

[Bank of America — Quantitative Finance Analyst]
  ...ata stores and big data environments. Hands-on expertise with data technologies and computing frameworks including but not limited to, Python, Spark, Airflow, Javascript and SQL. Ability to research new data technologies, architect novel data solutions for business problems and prove design approach throug...

[Barclays — AVP - Data Scientist & AI Engineer]
  ...be expected to work hands-on across on prem & cloud Compliance AI Platform. This requires strong proficiency in data ingestion and processing (Kafka, Airflow, Spark), cloud-based data platforms (Databricks, Snowflake, AWS S3/Azure), and SQL-based data transformation (dbt, PySpark). The candidate must demon...

[BlackRock — Data Engineer – AWS Native Data Platforms, Vice President]
  ...pelines for batch and event‑driven workloads, with a focus on reliability, scalability, and security. Develop and operate data workflows using Apache Airflow for orchestration and Python and S

In [13]:
quotes("control_m", limit=3)

In [14]:
quotes("dagster", limit=3)

[Barclays — Data Engineer - Analyst]
  ...ood communication skills (both verbal and written) Desirable skills/Preferred Qualifications: • Experience with orchestration tools (Snowflake Tasks, Dagster, Prefect) • Experience in data visualization and data analytics infra using Tableau (or similar BI tool) will be a value add • Past experience with l...



[BlackRock — Senior Data Architect/Data Engineer, Aladdin Engineering - Vice President]
  ...s layer approach. Exposure to orchestration tools (e.g., Directed acyclic graph-based workflow orchestration framework for data and batch processing, Dagster, Prefect) and patterns for dependency management and backfills. Streaming and event-driven data experience (e.g., distributed event streaming and mes...

[Brevan Howard — Front Office Data Engineer]
  ...perience working with market data providers such as Bloomberg, Refinitiv or ICE. Experience with orchestration frameworks such as Airflow, Prefect or Dagster. Experience building internal tools or dashboards using Dash, Streamlit or similar frameworks. Experience with GoldenSource or other enterprise data ...



## Collection history

One row per run. Several runs are needed before any trend claim is available;
`observations.csv` is built so the same tables can be recomputed per run.

In [15]:
runs_table = (
    pd.read_csv(d["data_dir"] / "run_log.csv")
    .groupby("run_date")
    .agg(firms=("firm", "nunique"), postings=("n_in_run", "sum"), new=("new", "sum"))
)
runs_table

,firms,postings,new
run_date,,,
2026-09-25,28,1993,1993
2026-09-27,64,7655,5664


## Where the language model is, and is not

`src/discover_sources.py` asks Claude exactly one question per firm: which
applicant-tracking system, and what is the endpoint? That is genuinely fuzzy
work — one firm's board token is `wehrtyou`, another's appears only inside JSON
embedded in an unrelated page, a third's differs from the obvious guess by two
characters and the obvious guess returns an empty board.

The answer is then validated by calling the endpoint and requiring real postings
back before anything is written to the manifest.

**Claude never reads a posting, never extracts a feature and never produces a
number.** Every count here comes from the declarative regexes above. The
proposer does not have to be trusted, because the gate confers the trust: a
hand-written proposal clears exactly the same check.

That boundary generalises well beyond job postings. Use the model where judgment
is cheap to check and expensive to automate, and keep it away from anything you
intend to report as a measurement.